# Delta Lake Schema Evolution

This notebook demonstrates Delta Lake's schema evolution capabilities. We'll:

1. Configure Spark for auto schema merging
2. Add new columns to the Delta table (e.g., "Discount", "Customer_Feedback")
3. Modify data types of existing columns safely
4. Run queries before and after schema changes
5. Continue streaming with the new schema
6. Show how Delta Lake handles missing columns in streaming data
7. Implement schema enforcement and schema validation
8. Compare with traditional database schema migration approaches

## 1. Initialize Spark Session with Delta Lake

In [ ]:
import os
import sys
import time
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, ArrayType
from pyspark.sql.functions import col, expr, lit, current_timestamp, array, struct
from delta.tables import DeltaTable

# Add scripts directory to path
sys.path.append('/opt/spark/scripts')
import utils

# Create Spark session with Delta Lake support and schema auto merge enabled
spark = SparkSession.builder \
    .appName("Delta Lake Schema Evolution") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

# Verify schema auto merge is enabled
auto_merge_enabled = spark.conf.get("spark.databricks.delta.schema.autoMerge.enabled")
print(f"Schema auto merge enabled: {auto_merge_enabled}")

## 2. Load Delta Table and Examine Current Schema

In [ ]:
# Define Delta table path
delta_table_path = "/opt/spark/data/processed/global_superstore_delta"

# Load the Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# Get initial table metrics
initial_metrics = utils.log_delta_table_metrics(spark, delta_table_path)
initial_version = initial_metrics["current_version"]

print(f"Initial table version: {initial_version}")
print(f"Initial record count: {initial_metrics['record_count']}")

# Examine current schema
current_df = spark.read.format("delta").load(delta_table_path)
print("\nCurrent schema:")
current_df.printSchema()

## 3. Add New Columns to the Delta Table

In [ ]:
# Create a DataFrame with new columns
print("Adding new columns to the Delta table...")

# Sample a subset of the data to modify
sample_df = current_df.limit(100)

# Add new columns
updated_df = sample_df \
    .withColumn("Customer_Feedback", lit("Satisfied")) \
    .withColumn("Delivery_Rating", lit(4)) \
    .withColumn("Tags", array(lit("retail"), lit("global"))) \
    .withColumn("Last_Updated", current_timestamp()) \
    .withColumn("Schema_Version", lit("v1"))

# Write back to the Delta table
updated_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("replaceWhere", "Row ID <= 100") \
    .save(delta_table_path)

# Examine updated schema
updated_schema_df = spark.read.format("delta").load(delta_table_path)
print("\nUpdated schema with new columns:")
updated_schema_df.printSchema()

# Show sample data with new columns
print("\nSample data with new columns:")
updated_schema_df \
    .select("Order ID", "Customer_Feedback", "Delivery_Rating", "Tags", "Schema_Version") \
    .filter(col("Customer_Feedback").isNotNull()) \
    .show(5)

## 4. Modify Data Types of Existing Columns

In [ ]:
# Check current data types
print("Current data types:")
spark.sql(f"DESCRIBE TABLE delta.`{delta_table_path}`").filter(col("col_name").isin(["Quantity", "Sales", "Profit"])).show()

# Sample data to modify
sample_df = spark.read.format("delta").load(delta_table_path).limit(100)

# Widen data types (safe conversion)
widened_df = sample_df \
    .withColumn("Quantity", col("Quantity").cast("long")) \
    .withColumn("Sales", col("Sales").cast("decimal(18,2)")) \
    .withColumn("Profit", col("Profit").cast("decimal(18,2)")) \
    .withColumn("Schema_Version", lit("v2"))

# Write back to the Delta table
print("\nModifying data types of existing columns...")
widened_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("replaceWhere", "Row ID <= 100") \
    .save(delta_table_path)

# Check updated data types
print("\nUpdated data types:")
spark.sql(f"DESCRIBE TABLE delta.`{delta_table_path}`").filter(col("col_name").isin(["Quantity", "Sales", "Profit"])).show()

## 5. Add Complex Nested Structure

In [ ]:
# Sample data to modify
sample_df = spark.read.format("delta").load(delta_table_path).limit(100)

# Add complex nested structure
nested_df = sample_df \
    .withColumn("Customer_Details", struct(
        col("Customer ID").alias("ID"),
        col("Customer Name").alias("Name"),
        col("Segment").alias("Segment"),
        struct(
            col("City").alias("City"),
            col("State").alias("State"),
            col("Country").alias("Country"),
            col("Postal Code").alias("Postal_Code")
        ).alias("Address")
    )) \
    .withColumn("Schema_Version", lit("v3"))

# Write back to the Delta table
print("Adding complex nested structure...")
nested_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("replaceWhere", "Row ID <= 100") \
    .save(delta_table_path)

# Examine updated schema with nested structure
updated_nested_df = spark.read.format("delta").load(delta_table_path)
print("\nUpdated schema with nested structure:")
updated_nested_df.printSchema()

# Show sample data with nested structure
print("\nSample data with nested structure:")
updated_nested_df \
    .select("Order ID", "Customer_Details") \
    .filter(col("Customer_Details").isNotNull()) \
    .show(5, truncate=False)

## 6. Query Data with Schema Evolution

In [ ]:
# Query data with new columns
print("Querying data with new columns:")
spark.sql(f"""
SELECT 
    `Order ID`, 
    Sales, 
    Profit, 
    Customer_Feedback, 
    Delivery_Rating,
    Schema_Version
FROM delta.`{delta_table_path}`
WHERE Customer_Feedback IS NOT NULL
LIMIT 5
""").show()

# Query data with nested structure
print("\nQuerying data with nested structure:")
spark.sql(f"""
SELECT 
    `Order ID`, 
    Customer_Details.Name as Customer_Name,
    Customer_Details.Address.City as City,
    Customer_Details.Address.Country as Country
FROM delta.`{delta_table_path}`
WHERE Customer_Details IS NOT NULL
LIMIT 5
""").show()

# Count records by schema version
print("\nCounting records by schema version:")
spark.sql(f"""
SELECT 
    Schema_Version, 
    COUNT(*) as record_count
FROM delta.`{delta_table_path}`
GROUP BY Schema_Version
ORDER BY Schema_Version
""").show()

## 7. Simulate Streaming with Schema Evolution

In [ ]:
# Create a streaming source directory
streaming_schema_dir = "/opt/spark/data/streaming_schema_evolution"
checkpoint_dir = "/opt/spark/data/checkpoints/schema_evolution"

# Ensure directories exist
os.makedirs(streaming_schema_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

# Generate streaming data with original schema
print("Generating streaming data with original schema...")
original_schema_data = utils.generate_test_data(num_records=50, scenario='normal')
original_schema_df = spark.createDataFrame(original_schema_data)
original_schema_df = original_schema_df.withColumn("Source", lit("stream_original_schema"))

# Write to CSV for streaming source
original_schema_path = os.path.join(streaming_schema_dir, "batch_original_schema.csv")
original_schema_df.toPandas().to_csv(original_schema_path, index=False)
print(f"Wrote original schema data to {original_schema_path}")

# Generate streaming data with new schema
print("\nGenerating streaming data with new schema...")
new_schema_data = utils.generate_test_data(num_records=50, scenario='schema_change')
new_schema_df = spark.createDataFrame(new_schema_data)
new_schema_df = new_schema_df \
    .withColumn("Source", lit("stream_new_schema")) \
    .withColumn("Customer_Feedback", lit("Good")) \
    .withColumn("Delivery_Rating", lit(5))

# Write to CSV for streaming source
new_schema_path = os.path.join(streaming_schema_dir, "batch_new_schema.csv")
new_schema_df.toPandas().to_csv(new_schema_path, index=False)
print(f"Wrote new schema data to {new_schema_path}")

# Set up streaming read
print("\nSetting up streaming read...")
streaming_df = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(streaming_schema_dir)

# Write stream to Delta table
print("Starting streaming write with schema evolution...")
query = streaming_df \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_dir) \
    .option("mergeSchema", "true") \
    .start(delta_table_path)

# Wait for the stream to process the data
time.sleep(10)
query.stop()
print("Streaming write completed")

# Check the results
print("\nChecking streaming results:")
spark.sql(f"""
SELECT 
    Source, 
    COUNT(*) as record_count,
    COUNT(Customer_Feedback) as feedback_count
FROM delta.`{delta_table_path}`
WHERE Source IN ('stream_original_schema', 'stream_new_schema')
GROUP BY Source
""").show()

## 8. Schema Enforcement and Validation

In [ ]:
# Demonstrate schema enforcement
print("Demonstrating schema enforcement...")

# Create a DataFrame with incompatible schema
incompatible_data = {
    "Order ID": ["TEST-001", "TEST-002", "TEST-003"],
    "Sales": [100.0, 200.0, 300.0],
    "Quantity": ["one", "two", "three"]  # String instead of integer/long
}
incompatible_df = spark.createDataFrame(pd.DataFrame(incompatible_data))

# Try to write with schema enforcement enabled
try:
    print("Attempting to write data with incompatible schema (should fail)...")
    incompatible_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "false") \
        .save(delta_table_path)
    print("Write succeeded (unexpected)")
except Exception as e:
    print(f"Write failed as expected: {str(e)[:200]}...")

# Now try with schema evolution enabled
print("\nAttempting to write with schema evolution enabled...")
compatible_data = {
    "Order ID": ["TEST-004", "TEST-005", "TEST-006"],
    "Sales": [400.0, 500.0, 600.0],
    "Quantity": [4, 5, 6],
    "New_Feature": ["Feature1", "Feature2", "Feature3"]  # New column
}
compatible_df = spark.createDataFrame(pd.DataFrame(compatible_data))

try:
    compatible_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .save(delta_table_path)
    print("Write succeeded with new column")
    
    # Verify the new column was added
    new_schema_df = spark.read.format("delta").load(delta_table_path)
    print("\nVerifying new column was added:")
    new_schema_df.filter(col("New_Feature").isNotNull()).select("Order ID", "Sales", "Quantity", "New_Feature").show()
except Exception as e:
    print(f"Write failed: {str(e)[:200]}...")

## 9. Compare with Traditional Database Schema Migration

In [ ]:
print("Comparison with Traditional Database Schema Migration:")
print("\nTraditional Relational Database:")
print("1. Schema changes typically require ALTER TABLE statements")
print("2. Adding columns often requires table locks or downtime")
print("3. Changing column types may require table rebuilds")
print("4. Complex migrations need careful planning and execution")
print("5. Often requires separate migration scripts and versioning")

print("\nDelta Lake Schema Evolution:")
print("1. Schema changes are handled automatically with 'mergeSchema' option")
print("2. Adding columns is a metadata-only operation, no data movement")
print("3. Widening column types is supported without data conversion")
print("4. No downtime required for schema changes")
print("5. Schema history is tracked in the transaction log")

print("\nBenefits of Delta Lake Schema Evolution:")
print("1. Reduced operational complexity for schema changes")
print("2. Continuous operation during schema evolution")
print("3. Backward compatibility for existing queries")
print("4. Support for complex nested structures")
print("5. Schema enforcement to prevent data corruption")

## 10. Examine Delta Table History

In [ ]:
# Examine Delta table history
history = delta_table.history(10).toPandas()
print("Delta table history (last 10 versions):")
history

## 11. Summary

In this notebook, we've demonstrated Delta Lake's schema evolution capabilities:

1. Adding new columns to an existing Delta table
2. Modifying data types of existing columns
3. Adding complex nested structures
4. Handling schema evolution in streaming workloads
5. Implementing schema enforcement and validation
6. Comparing with traditional database schema migration approaches

Delta Lake's schema evolution capabilities make it much easier to adapt to changing data requirements without disrupting existing processes. This is particularly valuable in agile development environments where data models evolve frequently.